In [ ]:
!pip install panel jupyter_bokeh --quiet

In [ ]:
import requests
import os
import json

IBM_CLOUD_API_KEY = "SNr7ju4ANn1r6CcuIpYnjCTeap3P0i5CchDOyPgGw259"

def get_iam_token(api_key: str) -> str:
    headers = {"Content-Type": "application/x-www-form-urlencoded", "Accept": "application/json"}
    data = {"grant_type": "urn:ibm:params:oauth:grant-type:apikey", "apikey": api_key}
    response = requests.post("https://iam.cloud.ibm.com/identity/token", headers=headers, data=data)
    response.raise_for_status()
    token_data = response.json()
    return token_data['access_token']

prompt_template = "You are a specialized healthcare assistant. Your persona is empathetic, professional, clear, and reassuring."

def recreate_history(history):
    result = []
    history = [item for item in history if isinstance(item, str)]
    if len(history) % 2 == 0:
        history.append("An error has occurred. Disregard irrelevant parts of the history")
    for idx, message in enumerate(history):
        if idx % 2 == 0:
            if idx == 0:
                result.append({
                    "role": "user",
                    "content": [{"type": "text", "text": f"{prompt_template} Input: {message}"}]
                })
            else:
                result.append({
                    "role": "user",
                    "content": [{"type": "text", "text": message}]
                })
        else:
            result.append({
                "role": "assistant",
                "content": message
            })
    return result

def call_watsonx_model_requests(prompt: str, history: list[str] = []) -> str:
    access_token = get_iam_token(IBM_CLOUD_API_KEY)
    url = "https://us-south.ml.cloud.ibm.com/ml/v1/text/chat?version=2023-05-29"
    body = {
      "messages": recreate_history(history + [prompt]),
      "project_id": "1955f8c1-81e7-46f6-bfa3-a921c1ecf344",
      "model_id": "ibm/granite-3-3-8b-instruct",
      "frequency_penalty": 0,
      "max_tokens": 2000,
      "presence_penalty": 0,
      "temperature": 0,
      "top_p": 1
    }
    headers = {
      "Accept": "application/json",
      "Content-Type": "application/json",
      "Authorization": f"Bearer {access_token}"
    }
    response = requests.post(
      url,
      headers=headers,
      json=body
    )
    if response.status_code != 200:
      raise Exception("Non-200 response: " + str(response.text))
    data = response.json()
    return data['choices'][0]['message']['content']


if __name__ == "__main__":
    my_prompt = "How do I deal with a stuffy nose?"
    print(f"Calling watsonx.ai with prompt: {my_prompt}")
    model_response = call_watsonx_model_requests(my_prompt)
    print(f"Prompt: {my_prompt}")
    print(f"Response: {model_response}")


Calling watsonx.ai with prompt: How do I deal with a stuffy nose?
Prompt: How do I deal with a stuffy nose?
Response: I'm sorry to hear that you're feeling uncomfortable. A stuffy nose can indeed be quite bothersome. Here are some suggestions to help alleviate the discomfort:

1. **Stay Hydrated**: Drink plenty of fluids, such as water, herbal tea, or clear broths. This can help thin your mucus, making it easier to drain.

2. **Use a Humidifier**: Dry air can exacerbate a stuffy nose. A humidifier can add moisture to the air, which may provide relief.

3. **Warm Compress**: Apply a warm compress to your face a few times a day. The heat can help open up nasal passages and reduce inflammation.

4. **Saline Spray**: Over-the-counter saline nasal sprays can help to moisten and flush out your nasal passages.

5. **Spicy Foods**: Consuming spicy foods like chili peppers may help to clear your nose by stimulating the runny nose reflex.

6. **Rest**: Ensure you're getting plenty of rest. Your 

In [ ]:
import requests
import os
import json
import panel as pn
import uuid
import time
import asyncio


pn.extension()
IBM_CLOUD_API_KEY = "SNr7ju4ANn1r6CcuIpYnjCTeap3P0i5CchDOyPgGw259"


def get_iam_token(api_key: str) -> str:
    headers = {"Content-Type": "application/x-www-form-urlencoded", "Accept": "application/json"}
    data = {"grant_type": "urn:ibm:params:oauth:grant-type:apikey", "apikey": api_key}
    response = requests.post("https://iam.cloud.ibm.com/identity/token", headers=headers, data=data)
    response.raise_for_status()
    return response.json()['access_token']


def recreate_history(history):
    result = []
    history = [item for item in history if isinstance(item, str)]
    if len(history) % 2 == 0:
        history.append("An error has occurred. Disregard irrelevant parts of the history")
    for idx, message in enumerate(history):
        if idx % 2 == 0:
            if idx == 0:
                result.append({
                    "role": "user",
                    "content": [{"type": "text", "text": f"Input: {message}"}]
                })
            else:
                result.append({
                    "role": "user",
                    "content": [{"type": "text", "text": message}]
                })
        else:
            result.append({
                "role": "assistant",
                "content": message
            })
    return result

def call_watsonx_model_requests(prompt: str, history: list[str] = [], sdoh_risk_factors: list[str] = [], prompt_template = "You are a specialized healthcare assistant. Your persona is empathetic, professional, clear, and reassuring. Factor in the SDOH responses in your response, but make sure to respond to the user first.") -> str:
    access_token = get_iam_token(IBM_CLOUD_API_KEY)
    url = "https://us-south.ml.cloud.ibm.com/ml/v1/text/chat?version=2023-05-29"

    sdoh_context = ""
    if sdoh_risk_factors:
        sdoh_context = "The user has identified the following social determinants of health risk factors: "
        sdoh_context += "; ".join(sdoh_risk_factors) + ". Please consider these factors when providing your advice and recommendations.\n\n"
        sdoh_context += "The user provided the risk factors as part of an automated system and, during your later interactions, will tell the reason they are reaching out. Disregard it initially, and only factor it in later on."

    final_prompt_for_model = prompt
    if sdoh_context:
        final_prompt_for_model = f"{sdoh_context}\nUser Prompt: {prompt}\n\nMake sure to respond to the user prompt."

    body = {
      "messages": recreate_history(history + [final_prompt_for_model]),
      "project_id": "1955f8c1-81e7-46f6-bfa3-a921c1ecf344",
      "model_id": "ibm/granite-3-3-8b-instruct",
      "frequency_penalty": 0,
      "max_tokens": 2000,
      "presence_penalty": 0,
      "temperature": 0,
      "top_p": 1
    }
    headers = {
      "Accept": "application/json",
      "Content-Type": "application/json",
      "Authorization": f"Bearer {access_token}"
    }
    response = requests.post(
      url,
      headers=headers,
      json=body
    )
    if response.status_code != 200:
      raise Exception("Non-200 response: " + str(response.text))
    data = response.json()
    return data['choices'][0]['message']['content']

SDOH_QUESTIONS = [
    "Do you have a safe and stable place to live? (Yes, No)",
    "Are you worried about losing your housing in the near future? (Yes, No)",
    "Are you able to afford healthy and nutritious food for yourself and your family? (Yes, No)",
    "Do you ever run out of food before you have money to buy more? (Yes, No)",
    "Do you have reliable transportation to your medical appointments, work, or school? (Yes, No)",
    "Is lack of transportation a barrier for you to access essential services? (Yes, No)",
    "What is your highest level of education completed?",
    "Do you feel your education adequately prepares you for opportunities you desire? (Yes, No)",
    "Are you currently employed? If not, are you looking for employment? (Yes, No)",
    "Does your employment provide you with stable income and benefits? (Yes, No)",
    "Do you have friends or family who can support you when needed? (Yes, No)",
    "Do you feel isolated or lack social connections? (Yes, No)",
    "Do you have difficulty paying for utilities like electricity, heating, or water? (Yes, No)",
    "Do you have access to affordable childcare if you need it? (Yes, No)",
    "Do you feel safe in your home and neighborhood? (Yes, No)",
    "Do you feel safe in your work? (Yes, No)"
]


def identify_sdoh_risk_factors(sdoh_responses: list[dict]) -> list[str]:
    risk_factors = []
    sdoh_responses = [item for item in sdoh_responses if isinstance(item, dict)]
    for response_dict in sdoh_responses:
        for question, answer in response_dict.items():
            normalized_answer = answer.lower().strip()
            if "safe and stable place to live" in question.lower():
                if "no" in normalized_answer or "not safe" in normalized_answer or "unstable" in normalized_answer:
                    risk_factors.append("Housing instability: unsafe or unstable living situation")
            elif "worried about losing your housing" in question.lower():
                if "yes" in normalized_answer or "worried" in normalized_answer:
                    risk_factors.append("Housing instability: worried about losing housing")
            # Food Security
            elif "afford healthy and nutritious food" in question.lower():
                if "no" in normalized_answer or "difficult" in normalized_answer or "struggle" in normalized_answer:
                    risk_factors.append("Food insecurity: difficulty affording healthy food")
            elif "run out of food" in question.lower():
                if "yes" in normalized_answer or "often" in normalized_answer:
                    risk_factors.append("Food insecurity: runs out of food before getting more")
            # Transportation Access
            elif "reliable transportation" in question.lower():
                if "no" in normalized_answer or "unreliable" in normalized_answer or "public transport is difficult" in normalized_answer:
                    risk_factors.append("Transportation barrier: unreliable transportation")
            elif "lack of transportation a barrier" in question.lower():
                if "yes" in normalized_answer or "a barrier" in normalized_answer:
                    risk_factors.append("Transportation barrier: lack of transportation impacts access")
            # Education
            elif "education adequately prepares you" in question.lower():
                if "no" in normalized_answer or "not adequately" in normalized_answer or "unprepared" in normalized_answer:
                    risk_factors.append("Education barrier: feels unprepared for opportunities")
            # Employment/Economic Stability
            elif "currently employed" in question.lower():
                if "no" in normalized_answer and "not looking" in normalized_answer:
                    risk_factors.append("Employment instability: unemployed and not seeking employment")
                elif "no" in normalized_answer and "unable" in normalized_answer:
                     risk_factors.append("Employment instability: unemployed due to inability to work")
            elif "employment provide you with stable income and benefits" in question.lower():
                if "no" in normalized_answer or "unstable" in normalized_answer or "not enough" in normalized_answer or "no benefits" in normalized_answer:
                    risk_factors.append("Economic instability: unstable income or lack of benefits")
            elif "difficulty paying for utilities" in question.lower():
                if "yes" in normalized_answer or "often" in normalized_answer or "difficult" in normalized_answer:
                    risk_factors.append("Economic instability: difficulty paying utilities")
            # Social Support
            elif "friends or family who can support you" in question.lower():
                if "no" in normalized_answer or "few" in normalized_answer or "not really" in normalized_answer:
                    risk_factors.append("Social isolation: lacks social support network")
            elif "feel isolated or lack social connections" in question.lower():
                if "yes" in normalized_answer or "isolated" in normalized_answer or "lack connections" in normalized_answer:
                    risk_factors.append("Social isolation: feels isolated")
            # Personal Safety
            elif "feel safe in your home and neighborhood" in question.lower():
                if "no" in normalized_answer or "unsafe" in normalized_answer or "not safe" in normalized_answer:
                    risk_factors.append("Personal safety concern: feels unsafe in home or neighborhood")
            # Childcare Access
            elif "access to affordable childcare" in question.lower():
                if "no" in normalized_answer or "difficult" in normalized_answer or "expensive" in normalized_answer:
                    risk_factors.append("Childcare barrier: lack of affordable childcare access")
    return list(set(risk_factors))


async def panel_chat_callback(
    contents: str,
    user: str,
    instance: pn.chat.ChatInterface
):
    if 'session_id' not in pn.state.cache:
        pn.state.cache['session_id'] = str(uuid.uuid4())
        print(f"New Panel session started: {pn.state.cache['session_id']}")
    # Initialize session state variables for SDOH screening and original query
    if 'sdoh_responses' not in pn.state.cache:
        pn.state.cache['sdoh_responses'] = []
    if 'deferred_initial_query' not in pn.state.cache:
        pn.state.cache['deferred_initial_query'] = None
    if 'identified_sdoh_risks' not in pn.state.cache:
        pn.state.cache['identified_sdoh_risks'] = []
    current_sdoh_question_index = len(pn.state.cache['sdoh_responses'])
    sdoh_responses = pn.state.cache['sdoh_responses']
    # Scenario 1: Initiating SDOH screening
    # This happens if it's the very first message in a new session (or after a full reset)
    if current_sdoh_question_index < 1:
        pn.state.cache['deferred_initial_query'] = contents
        pn.state.cache['sdoh_responses'].append(SDOH_QUESTIONS[0])
        return f"""Hello! Before I can provide the best healthcare assistance, I need to conduct a brief risk assessment screening to better understand your needs. Your responses are confidential.\n\n{len(pn.state.cache['sdoh_responses'])}. {SDOH_QUESTIONS[0]}"""
    elif 0 < current_sdoh_question_index < len(SDOH_QUESTIONS) - 1:
        pn.state.cache['sdoh_responses'].append({SDOH_QUESTIONS[current_sdoh_question_index - 1]: contents})
        next_question = SDOH_QUESTIONS[len(pn.state.cache['sdoh_responses'])-1]
        return f"{len(pn.state.cache['sdoh_responses'])}. {next_question}"
    else:
        if current_sdoh_question_index == len(SDOH_QUESTIONS):
            pn.state.cache['sdoh_responses'].append({SDOH_QUESTIONS[current_sdoh_question_index - 1]: contents})
        identified_risks = identify_sdoh_risk_factors(sdoh_responses)
        pn.state.cache['identified_sdoh_risks'] = identified_risks
        if pn.state.cache.get('deferred_initial_query', None):
            final_user_query_to_process = pn.state.cache['deferred_initial_query']
            pn.state.cache['deferred_initial_query'] = None
        else:
            final_user_query_to_process = contents
        if 'session_history' not in pn.state.cache:
            pn.state.cache['session_history'] = []
        chat_history = pn.state.cache['session_history']
        model_reply = call_watsonx_model_requests(final_user_query_to_process, chat_history, sdoh_risk_factors=identified_risks)
        chat_history.extend([final_user_query_to_process, model_reply])
        return f"SDOH Screening Complete! Here's a response to your initial query, considering your situation:\n{'; '.join(identified_risks)}\n\n{model_reply}"

chatbot_ui = pn.chat.ChatInterface(
    callback=panel_chat_callback,
    callback_user="Healthcare Chatbot",
    height=500,
    auto_scroll_limit=100,
    callback_exception='verbose'
)

def new_user_session_callback(event):
    pn.state.cache.pop('session_id', None)
    pn.state.cache.pop('sdoh_responses', None)
    pn.state.cache.pop('deferred_initial_query', None)
    pn.state.cache.pop('identified_sdoh_risks', None)
    pn.state.cache.pop('session_history', None)
    chatbot_ui.clear()

new_user_button = pn.widgets.Button(
    name="New User",
    button_type="primary", # 'primary' (blue), 'success', 'warning', 'danger'
    icon="user-plus"  # Adds a nice icon
)
new_user_button.on_click(new_user_session_callback)
app_layout = pn.Column(
    pn.pane.Markdown("## Healthcare Chatbot (Watsonx)"),
    pn.pane.Markdown("Ask healthcare-related questions to the deployed Watsonx model."),
    new_user_button,
    chatbot_ui,
    max_width=800,
    sizing_mode="stretch_width"
)

app_layout

Column(max_width=800, sizing_mode='stretch_width')
    [0] Markdown(str)
    [1] Markdown(str)
    [2] Button(button_type='primary', icon='user-plus', name='New User')
    [3] ChatInterface(_button_data={'send': _ChatButtonData(i...}, _buttons={'send': Button(align='cen...}, _input_container=Row, _input_layout=Row, _placeholder=ChatMessage, _widgets={'ChatAreaInput': ChatArea...}, auto_scroll_limit=100, callback=<function panel_chat_callb..., callback_exception='verbose', callback_user='Healthcare Chatbot', height=500, show_button_name=True, sizing_mode='stretch_width', widgets=[ChatAreaInput(css_classes...])

New Panel session started: c4aed36f-d850-45b3-aea8-20bc2afcf810
